In [1]:
# ============================
# HELPERS 
# ============================

def process_vessel_name(name):
    if not isinstance(name, str):
        return None, False, False
    
    original = name.strip()
    s = original.upper()
    
    # Flags
    is_oos = bool(re.search(r"\bOOS\b", s))
    is_optional = bool(re.search(r"\bOR\b", s))
    
    # Remove parentheses
    clean = re.sub(r"\(.*?\)", "", original)
    
    # Remove PMAX / OS / OOS text but DO NOT remove "or"
    clean = re.sub(r"\bPMAX\b", "", clean, flags=re.IGNORECASE)
    clean = re.sub(r"\bOS\b", "", clean, flags=re.IGNORECASE)
    clean = re.sub(r"\bOOS\b", "", clean, flags=re.IGNORECASE)
    
    clean = re.sub(r"\s{2,}", " ", clean).strip()
    
    return clean.title(), is_oos, is_optional

import re

# Words we ALWAYS keep uppercase
ALWAYS_UPPER = {
    "BW", "HLS", "AGT", "TC", "LPG", "E1"
}

def smart_title_word(word):
    w = word.upper()

    # Keep acronyms (2-3 letters) uppercase
    if w in ALWAYS_UPPER:
        return w
    
    if len(word) <= 3 and word.isupper():
        return word.upper()

    return word.capitalize()


def format_vessel_name(name):
    if not isinstance(name, str):
        return None
    
    original = name.strip()

    # Remove parentheses
    s = re.sub(r"\(.*?\)", "", original)

    # Remove suffix keywords
    s = re.sub(r"\bPMAX\b", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\bOS\b", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\bOOS\b", "", s, flags=re.IGNORECASE)

    s = re.sub(r"\s{2,}", " ", s).strip()

    words = s.split()
    formatted = " ".join(smart_title_word(w) for w in words)

    return formatted


In [2]:
import pandas as pd

file_path = "All_Brokers_All_Emails_Combined.xlsx"
sheets = pd.read_excel(file_path, sheet_name=None)

import re


standard_rows = []

for name, df in sheets.items():

    # ============================
    # BRS FIXTURES (USG only)
    # ============================
    if "BRS_FIXTURES" in name:

        df_usg = df[df["Route"] == "USG"]

        for _, r in df_usg.iterrows():
            standard_rows.append({
                "Broker": r.get("Broker"),
                "Vessel": r.get("Vessel"),
                "Counterparty": r.get("Charterer"),
                "Route": r.get("Route"),
                "ETA Start": r.get("Laycan Start"),
                "ETA End": r.get("Laycan End"),
                "ETA Midpoint": r.get("Laycan Midpoint"),
                "Notes": r.get("Raw Line"),
                "Email Sent Date": r.get("Email Sent Date")
            })

    # --------------------------
    # Fearnleys
    # --------------------------
    elif "Fearnleys" in name:

        for _, r in df.iterrows():
            standard_rows.append({
                "Broker": r.get("Broker"),
                "Vessel": r.get("Vessel"),
                "Counterparty": r.get("Owner"),
                "Route": "USG",
                "ETA Start": r.get("ETA Start"),
                "ETA End": r.get("ETA End"),
                "ETA Midpoint": r.get("ETA Midpoint"),
                "Notes": r.get("Notes"),
                "Email Sent Date": r.get("Email Sent Date")
            })

    # --------------------------
    # Poten
    # --------------------------
    elif "Poten" in name:

        for _, r in df.iterrows():
            standard_rows.append({
                "Broker": r.get("Broker"),
                "Vessel": r.get("Vessel"),
                "Counterparty": r.get("Owner"),
                "Route": "USG",
                "ETA Start": r.get("ETA USG Start"),
                "ETA End": r.get("ETA USG End"),
                "ETA Midpoint": r.get("ETA USG Midpoint"),
                "Notes": r.get("Additional Comments"),
                "Email Sent Date": r.get("Email Sent Date")
            })

    # --------------------------
    # Gibson
    # --------------------------
    elif "Gibson" in name:

        for _, r in df.iterrows():
            standard_rows.append({
                "Broker": r.get("Broker"),
                "Vessel": r.get("Vessel"),
                "Counterparty": r.get("OWNER"),
                "Route": "USG",
                "ETA Start": r.get("ETA USG Start"),
                "ETA End": r.get("ETA USG End"),
                "ETA Midpoint": r.get("ETA USG Midpoint"),
                "Notes": r.get("COMMENTS"),
                "Email Sent Date": r.get("Email Sent Date")
            })

    # --------------------------
    # Affinity
    # --------------------------
    elif "Affinity" in name:

        for _, r in df.iterrows():
            standard_rows.append({
                "Broker": r.get("Broker"),
                "Vessel": r.get("Vessel"),
                "Counterparty": r.get("Control"),
                "Route": "USG",
                "ETA Start": r.get("ETA Start"),
                "ETA End": r.get("ETA End"),
                "ETA Midpoint": r.get("ETA Midpoint"),
                "Notes": r.get("Notes"),
                "Email Sent Date": r.get("Email Sent Date")
            })

# Build unified dataframe
standard_df = pd.DataFrame(standard_rows)

# Sort chronologically
standard_df = standard_df.sort_values("ETA Midpoint")

standard_df[["Vessel_Clean", "Is_OOS", "Is_Optional"]] = (
    standard_df["Vessel"]
    .apply(lambda x: pd.Series(process_vessel_name(x)))
)

standard_df["Vessel_Formatted"] = standard_df["Vessel"].apply(format_vessel_name)

print(standard_df.head())

       Broker                    Vessel Counterparty Route  ETA Start  \
52     Gibson                 BW BREEZE       BW LPG   USG 2026-03-14   
37  Fearnleys  BW Breeze or BW Magellan           BW   USG 2026-03-14   
67      Poten            BW Magellan os           BW   USG 2026-03-16   
0    Affinity                    BW TBN           BW   USG 2026-03-14   
17   Affinity                    BW TBN           BW   USG 2026-03-14   

      ETA End        ETA Midpoint  \
52 2026-03-15 2026-03-14 12:00:00   
37 2026-03-17 2026-03-15 12:00:00   
67 2026-03-17 2026-03-16 12:00:00   
0  2026-03-20 2026-03-17 00:00:00   
17 2026-03-20 2026-03-17 00:00:00   

                                          Notes Email Sent Date  \
52  VIA CAPE. OOS MAGELLAN 16-17 MAR VIA PANAMA      2026-02-26   
37                   via Cape / via Pan NB 12th      2026-02-24   
67                   Northbound Panama 12 March      2026-02-26   
0                                    via Panama      2026-02-27   
17 

In [ ]:
# ============================================
# Build "Events" sheet (Added/Removed) with
# - Source preference enforcement (and losers marked Removed (Preference))
# - Day-to-day Added/Removed based on ACTIVE (preferred) snapshot
#
# Preference order (best -> worst):
#   Clarksons/Gibsons  (rank 1)
#   Affinity           (rank 2)
#   Fearnleys          (rank 3)
#   Poten              (rank 4)
#
# Key behavior:
# - If ETA Start/End are identical (same day, same vessel/pos/controller),
#   keep best source; mark others as Removed (Preference), with Removed By = winner.
# - Then compute Added/Removed between days from the preferred snapshot ONLY.
#
# Output Excel:
#   Sheet "Raw"    = original standard_df
#   Sheet "Events" = events table
# ============================================

import pandas as pd
import numpy as np
import re

# ----------------------------
# Assumes you already have:
#   standard_df built earlier (your unified table)
#   standard_df has: Broker, Vessel_Formatted, Route, Counterparty,
#                   ETA Start, ETA End, ETA Midpoint, Notes, Email Sent Date
# ----------------------------


# --- 1) Dates / timestamps ---
standard_df["EmailSentTS"] = pd.to_datetime(standard_df["Email Sent Date"], errors="coerce")
standard_df["Date"] = standard_df["EmailSentTS"].dt.date

# --- 2) Preference ranking ---
PREF_RANK = {
    "Clarksons": 3,
    "Clarkson": 3,
    "Gibson": 3,
    "Gibsons": 3,
    "Affinity": 2,
    "Fearnleys": 2,
    "Poten": 2,
    "BRS": 1,

}
standard_df["PrefRank"] = standard_df["Broker"].map(PREF_RANK).fillna(99).astype(int)

standard_df["Vessel_Key"] = (
    standard_df["Vessel_Formatted"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()                 # <-- canonical
    .str.replace(r"\s+", " ", regex=True)
)

# --- 3) Normalise key columns to avoid duplicates (HELIOS vs Helios etc.) ---
standard_df["Expected Position"] = (
    standard_df.get("Route", "")
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

standard_df["Controller"] = (
    standard_df.get("Counterparty", "")
    .fillna("")
    .astype(str)
    .str.strip()
    .str.title()
)

# Vessel formatting already done earlier; just ensure string
standard_df["Vessel_Formatted"] = (
    standard_df.get("Vessel_Formatted", standard_df.get("Vessel", ""))
    .fillna("")
    .astype(str)
    .str.strip()
)

standard_df["Notes"] = standard_df.get("Notes", "").fillna("").astype(str).str.strip()

# ETA date for the event output (use midpoint date)
standard_df["ETA"] = pd.to_datetime(standard_df.get("ETA Midpoint"), errors="coerce").dt.date

# Ballasting Route classifier from Notes
def classify_ballasting_route(notes: str) -> str:
    s = (notes or "").lower()
    has_pan = ("panama" in s) or re.search(r"\bpan\b", s) is not None
    has_cape = "cape" in s
    if has_pan and has_cape:
        return "Fluid"
    if has_pan:
        return "Panama"
    if has_cape:
        return "Cape"
    return ""

standard_df["Ballasting Route"] = standard_df["Notes"].apply(classify_ballasting_route)

# --- 4) Preference de-dupe within same day + identical ETA range, and emit Removed (Preference) ---
dedupe_key = [
    "Date",
    "Vessel_Key",
    "Expected Position",
    "Controller",
    "ETA Start",
    "ETA End",
]

std_sorted = standard_df.sort_values(["Date", "PrefRank", "EmailSentTS", "Broker"]).copy()

pref_events = []
winners = []

for _, grp in std_sorted.groupby(dedupe_key, dropna=False):
    grp = grp.sort_values(["PrefRank", "EmailSentTS", "Broker"])
    winner = grp.iloc[0]
    winners.append(winner)

    if len(grp) > 1:
        for i in range(1, len(grp)):
            loser = grp.iloc[i]
            pref_events.append({
                "Date": loser["Date"],
                "Vessel": loser["Vessel_Formatted"],
                "Expected Position": loser["Expected Position"],
                "Controller": loser["Controller"],
                "ETA": loser["ETA"],
                "Notes": loser["Notes"],
                "Source": loser["Broker"],
                "Logic": "Removed (Preference)",
                "Ballasting Route": loser["Ballasting Route"],
                "Removed By": winner["Broker"],
            })

active_daily = pd.DataFrame(winners).copy()

# --- 5) Build state keys for day-to-day diff from the preferred snapshot ---
def make_state_key(df: pd.DataFrame) -> pd.Series:
    def sdate(x):
        if pd.isna(x):
            return ""
        return pd.to_datetime(x).strftime("%Y-%m-%d")

    return (
        df["Vessel_Key"].fillna("").astype(str) + "||" +
        df["Expected Position"].fillna("").astype(str) + "||" +
        df["Controller"].fillna("").astype(str) + "||" +
        df["ETA Start"].apply(sdate) + "||" +
        df["ETA End"].apply(sdate)
    )

active_daily["StateKey"] = make_state_key(active_daily)

# --- 6) Diff snapshots day-by-day to generate Added/Removed events ---
events = []
dates = sorted(active_daily["Date"].dropna().unique())

prev_keys = set()
prev_rows = {}

for d in dates:
    snap = active_daily[active_daily["Date"] == d].copy()
    curr_keys = set(snap["StateKey"].tolist())

    added_keys = curr_keys - prev_keys
    removed_keys = prev_keys - curr_keys

    # Added today
    for _, r in snap[snap["StateKey"].isin(added_keys)].iterrows():
        events.append({
            "Date": r["Date"],
            "Vessel": r["Vessel_Formatted"],
            "Expected Position": r["Expected Position"],
            "Controller": r["Controller"],
            "ETA": r["ETA"],
            "Notes": r["Notes"],
            "Source": r["Broker"],
            "Logic": "Added",
            "Ballasting Route": r["Ballasting Route"],
            "Removed By": "",
        })

    # Removed since previous day
    for k in removed_keys:
        pr = prev_rows.get(k)
        if pr:
            events.append({
                "Date": d,
                "Vessel": pr["Vessel_Formatted"],
                "Expected Position": pr["Expected Position"],
                "Controller": pr["Controller"],
                "ETA": pr["ETA"],
                "Notes": pr["Notes"],
                "Source": pr["Broker"],
                "Logic": "Removed",
                "Ballasting Route": pr["Ballasting Route"],
                "Removed By": "",
            })

    prev_keys = curr_keys
    prev_rows = snap.set_index("StateKey").to_dict(orient="index")

# Combine preference-removals + day-diff events
events_df = pd.DataFrame(pref_events + events)

# --- 7) Final cleanup / formatting / dedupe ---
if not events_df.empty:
    events_df["Date"] = pd.to_datetime(events_df["Date"]).dt.strftime("%d/%m/%Y")
    events_df["ETA"] = pd.to_datetime(events_df["ETA"]).dt.strftime("%d/%m/%Y")

    events_df = events_df[
        ["Date", "Vessel", "Expected Position", "Controller", "ETA", "Notes",
         "Source", "Logic", "Ballasting Route", "Removed By"]
    ].sort_values(["Date", "Vessel", "Logic", "Source"])

    # safety dedupe (prevents accidental repeated rows)
    events_df = events_df.drop_duplicates(
        subset=["Date","Vessel","Expected Position","Controller","ETA","Source","Logic","Removed By"]
    )

# --- 8) Write Raw + Events to a single Excel with 2 sheets ---
out_path = "Single_Table_Combined.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    standard_df.to_excel(writer, sheet_name="Raw", index=False)
    events_df.to_excel(writer, sheet_name="Events", index=False)

print(f"[OK] Wrote: {out_path} (Raw + Events)")
print(events_df.head(20))

[OK] Wrote: Single_Table_Combined.xlsx (Raw + Events)
          Date                    Vessel Expected Position Controller  \
6   24/02/2026  BW Breeze Or BW Magellan               USG         Bw   
7   24/02/2026                BW Sirocco               USG        Stl   
8   24/02/2026               Continental               USG     Helios   
9   24/02/2026                  Gas Vela               USG    Sinogas   
10  24/02/2026                 Gas Virgo               USG     Wanhua   
11  24/02/2026                Helios TBN               USG     Helios   
12  24/02/2026                  Jia Yuan               USG     Keegan   
13  24/02/2026                 Kedarnath               USG       Pdec   
14  24/02/2026             Lycaste Peace               USG    Astomos   
15  24/02/2026             Oriental King               USG         Bw   
16  24/02/2026           Pacific Binzhou               USG      Shell   
17  24/02/2026      Pertamina Gas Caspia               USG        Scg 